# api adapter contract v1

验证 `src.api` 的 broker/data adapter contract：v1 只允许 paper/sim，现有 paper broker 和 paper market data adapter 必须满足稳定协议。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.api import (
    BrokerAdapter,
    MarketDataAdapter,
    PaperBrokerAdapter,
    PaperMarketDataAdapter,
    assert_paper_sim_mode,
)

CARD = Path('research/api_adapter_contract_v1')
CARD.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH = CARD / 'analysis_summary.json'
CONTRACT_PATH = CARD / 'adapter_contract_summary.json'

In [2]:
frame = pd.DataFrame(
    [
        {'date': '2026-01-02', 'instrument': 'SOXL', 'close': 10.0},
        {'date': '2026-01-05', 'instrument': 'SOXL', 'close': 12.0},
    ]
)
broker = PaperBrokerAdapter(cash=1000.0, shares=0)
market_data = PaperMarketDataAdapter.from_frame(frame, as_of='2026-01-05')
quote = market_data.latest_quote('SOXL')
snapshot = broker.place_target_position(target_position=0.5, price=quote.price)

contract_summary = {
    'schema_version': 'api.adapter_contract_summary.v1',
    'broker_protocol': 'BrokerAdapter',
    'market_data_protocol': 'MarketDataAdapter',
    'paper_broker_runtime_check': isinstance(broker, BrokerAdapter),
    'paper_market_data_runtime_check': isinstance(market_data, MarketDataAdapter),
    'allowed_modes': [assert_paper_sim_mode('paper'), assert_paper_sim_mode('sim')],
    'live_mode_rejected': False,
    'quote_price': quote.price,
    'snapshot_shares': snapshot.shares,
    'snapshot_position': snapshot.position,
}
try:
    assert_paper_sim_mode('live')
except ValueError:
    contract_summary['live_mode_rejected'] = True

assert contract_summary['paper_broker_runtime_check'] is True
assert contract_summary['paper_market_data_runtime_check'] is True
assert contract_summary['allowed_modes'] == ['paper', 'sim']
assert contract_summary['live_mode_rejected'] is True
assert contract_summary['quote_price'] == 12.0

CONTRACT_PATH.write_text(json.dumps(contract_summary, ensure_ascii=False, indent=2), encoding='utf-8')
contract_summary

{'schema_version': 'api.adapter_contract_summary.v1',
 'broker_protocol': 'BrokerAdapter',
 'market_data_protocol': 'MarketDataAdapter',
 'paper_broker_runtime_check': True,
 'paper_market_data_runtime_check': True,
 'allowed_modes': ['paper', 'sim'],
 'live_mode_rejected': True,
 'quote_price': 12.0,
 'snapshot_shares': 41,
 'snapshot_position': 0.492}

In [3]:
summary = {
    'schema_version': 'research.api_adapter_contract_v1.summary',
    'status': 'completed',
    'broker_protocol': contract_summary['broker_protocol'],
    'market_data_protocol': contract_summary['market_data_protocol'],
    'paper_broker_runtime_check': contract_summary['paper_broker_runtime_check'],
    'paper_market_data_runtime_check': contract_summary['paper_market_data_runtime_check'],
    'allowed_modes': contract_summary['allowed_modes'],
    'live_mode_rejected': contract_summary['live_mode_rejected'],
    'adapter_contract_summary': str(CONTRACT_PATH),
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.api_adapter_contract_v1.summary',
 'status': 'completed',
 'broker_protocol': 'BrokerAdapter',
 'market_data_protocol': 'MarketDataAdapter',
 'paper_broker_runtime_check': True,
 'paper_market_data_runtime_check': True,
 'allowed_modes': ['paper', 'sim'],
 'live_mode_rejected': True,
 'adapter_contract_summary': 'research/api_adapter_contract_v1/adapter_contract_summary.json'}

## 中文结论

- 做了什么：在 `src.api` 层验证 broker/data adapter contract，确认 `PaperBrokerAdapter` 满足 `BrokerAdapter` 协议，`PaperMarketDataAdapter` 满足 `MarketDataAdapter` 协议，并验证统一的 paper/sim mode guard。
- 核心指标：`paper_broker_runtime_check=True`、`paper_market_data_runtime_check=True`、允许模式为 `paper/sim`，`live` 模式被拒绝；样例 quote price 为 `12.0`，paper target position 后持仓为 `41` 股左右的半仓量级。
- regime 差异：本实验不做 regime 切片，只验证 API adapter 边界；regime/model 差异仍由 analysis/dashboard 层消费。
- 是否晋升：可作为 `src/api` 稳定 contract 使用；不晋升为真实 broker 接口，v1 仍只实现 paper/sim。
- 风险：Protocol 是结构化边界，不代表真实 broker 连接、权限隔离、成交回报、重连和审计已经实现；这些必须通过后续架构扩展决策进入。
- 下一步：真实 broker 适配若进入范围，应先实现独立 adapter 并通过同一 contract、healthcheck、intent ledger 和 paper/sim guardrail 审查。